활용 분석 방법 : 소셜 데이터, ML, DL(?), EDA, NLP, LLM 등 활용
주요 데이터 : 정형데이터를 확보하기 어려우므로, 소셜데이터를 주로 활용예정, API를 활용해 데이터를 끌어올 수도 있음
타겟 사이트 : 커뮤니티, 유튜브,,? 등 소셜 데이터 최대한 활용
주제 : 사회현상?, 소비현상? 등을 기반으로 이야기를 전개해 나감
블로그, 커뮤니티 등에서 뭔가를 얻을 수 있을 거 같은데.... 본문 댓글을 LLM에 보내서, 데이터 라벨링을 얻는다.?


Naver API
Client ID : 9q7w7kzlkz
Client Secret : 9oJbtNtakJ9vI2ipVbJkbco6YlJvorvzBF3nvUtG

In [1]:
import requests
import pandas as pd

In [20]:
API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/blog"
CLIENT_ID='9q7w7kzlkz'
CLIENT_SECRET_KEY='9oJbtNtakJ9vI2ipVbJkbco6YlJvorvzBF3nvUtG'
headers={'X-NCP-APIGW-API-KEY-ID':CLIENT_ID,
         'X-NCP-APIGW-API-KEY':CLIENT_SECRET_KEY}
QUERY="경기떡집"
DISPLAY="10" # 1~100 단위
START="1"
SORT="date"
FORMAT="json"
BATCH_SIZE=100
TARGET_NUMBER=1000
START_NUM=1
params={'query':QUERY,
        'display':BATCH_SIZE,
        'start':START_NUM,
        'sort':'date',
        'format':FORMAT}


In [22]:
def blog_extractor(start_num, target_number, batch_size):
    total_data=[]
    
    for start in range(start_num, target_number, batch_size):
        params={'query':QUERY,
        'display':BATCH_SIZE,
        'start':start,
        'sort':'date',
        'format':FORMAT}

        try:
            response=requests.get(
                API_URL,
                headers=headers,
                params=params,
                timeout=15
            )
            response.raise_for_status()
            items=response.json().get('items',[])
            
            if not items:
                break
            total_data.extend(items)
            start_num+=batch_size
        except Exception as e:
            print(e)
            break
    
    return total_data
    

In [17]:
[i for i in range(1,1000,100)]

[1, 101, 201, 301, 401, 501, 601, 701, 801, 901]

In [23]:
blog_extractor=blog_extractor(START_NUM, TARGET_NUMBER, BATCH_SIZE)

In [25]:
len(blog_extractor)

1000

In [9]:
df=pd.DataFrame(blog_extractor)

In [10]:
import re
def txtcleaner(txt):
    if not isinstance(txt, str):
        return ""
    
    txt=re.sub(r'[^가-힣\s]',' ', txt) # 가-힣과 공백을 제외한 모든 문제 제거하는 메서드
    txt=re.sub(r'\s+',' ', txt).strip() # 연소 공백, 줄바꿈 등을 공백 하나로 정리
    return txt
target_col=['title','description']
for col in target_col:
    df[col]=df[col].apply(txtcleaner)
    

In [11]:
drop_cols=['bloggername','description','bloggerlink'] # 블로그 이름은 의미가 없고, description 대신 본문을 크롤링 할 것이기 때문에, 지움. bloggerlink도 의미 없음
df=df.drop(columns=drop_cols)

In [12]:
df

,title,link,postdate
0,경기떡집 아침부터 줄 서는 집,https://blog.naver.com/insnow02/224420696695,20260923
1,서울 망원 추석 선물 경기떡집 떡 추천 마스코바도 딸기 호박 설,https://blog.naver.com/ram_m0ment/224420687622,20260923
2,구리 갈매 떡집 떡이유 딸기 블루베리 피자설기 오픈런 추천,https://blog.naver.com/kisses0j/224420673885,20260923
3,부천 범박동 아빠손 떡집 추석 송편맛집 가성비 부천 떡집 추천 떡,https://blog.naver.com/ejammy/224420659466,20260923
4,수원 떡집 우아당 떡집 송편만들기키트 만족후기,https://blog.naver.com/122girl/224420645770,20260923
...,...,...,...
195,자연을담은떡 수원 피자설기 파는 곳 정자시장 떡집 자연을담은떡,https://blog.naver.com/sjj926/224412321702,20260916
196,시흥 배곧 떡집 해와달떡 추석 선물 추천 가성비 떡집 아기 돌 답례,https://blog.naver.com/hj22223/224412936206,20260915
197,분당 맛집 분당 피자설기 허허 떡집 트만에 성공한 후기 트 초,https://blog.naver.com/roongzip/224412837262,20260915
198,생방송투데이 인절미 떡 전성시대 떡집 멜론설기 바나나꿀설기 크,https://blog.naver.com/eueuy/224412753150,20260915


In [66]:
df.drop(columns='domain', inplace=True)
df

,title,link,postdate
0,전어 다음은 대하다 월 말 꼭 먹어야 할 가을 제철 먹거리,https://blog.naver.com/leolove100/224419945013,20260922
1,강남 논현동 맛집 산초 간장게장 꽃게 무침,https://blog.naver.com/boyun1013/224419938393,20260922
2,가을 제철음식 아무 때나 먹으면 된다 맛있는 시기는 따로 있습니,https://blog.naver.com/pangyoung/224419928925,20260922
3,월부터 꼭 먹어야 한다 놓치면 아쉬운 가을 제철음식,https://blog.naver.com/fourseasons101/22441991...,20260922
4,짧글 다시 가고 싶어지는 영덕 해산물 갓성비 라면맛집 재방문 후,https://blog.naver.com/ghd34ghd43/224419918601,20260922
...,...,...,...
9995,꽃게 어부가 직접 판매하는 급냉 숫 꽃게 구매 전 확인사항,https://blog.naver.com/samwoo39-/224419650451,20260922
9996,은퇴후 밥상이야기 소고기와 꽃게 가 도착합니다,https://blog.naver.com/professionalmom/2244196...,20260922
9997,집에서 먹으니 더 푸짐했던 꽃게 찜,https://blog.naver.com/0516sweet/224419647655,20260922
9998,봄 꽃게 가을 꽃게 제철시기 부산자갈치 꽃게 맛집 내돈내산 추천,https://blog.naver.com/msy8013/224419643932,20260922


In [ ]:
# 블로그 HTML 구조를 파악한 뒤에 차례로 크롤링 해야 함
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
body_data=[]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}
def body_text_extractor(url):
    print('블로그 본문을 수집합니다.')
    try:
        # 1. 외부 네이버 블로그 페이지 요청
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        )
        response.raise_for_status()

        outer_soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # 2. 실제 본문을 담은 iframe 주소 찾기
        iframe = outer_soup.select_one("iframe#mainFrame")

        if iframe is not None and iframe.get("src"):
            inner_url = urljoin(url, iframe["src"])

            # 3. iframe 내부 페이지 재요청
            inner_response = requests.get(
                inner_url,
                headers=HEADERS,
                timeout=10
            )
            inner_response.raise_for_status()

            soup = BeautifulSoup(
                inner_response.text,
                "html.parser"
            )

        else:
            # iframe 없는 주소(PostView 등)는 현재 HTML 그대로 사용
            soup = outer_soup

        # 4. 최신 스마트에디터 본문 선택
        body_tag = soup.select_one("div.se-main-container")

        # 5. 구형 에디터 구조도 대비
        if body_tag is None:
            body_tag = soup.select_one("#postViewArea")

        if body_tag is None:
            print(f"본문 영역을 찾지 못함: {url}")
            return None

        print('네이버 블로그 본문 크롤링을 완료했습니다. ')
        
        time.sleep(3)
        return body_tag.get_text(
            separator="\n",
            strip=True)
    
    except requests.RequestException as e:
        print(f"요청 오류: {url} / {e}")
        return None

html에서 iframe 구조는 HTML 페이지 안에서 다른 웹 문서나 외부 서비스를 작은 독립 창처럼 삽입하기 위해서 쓰는 것으로, 다른페이지를 직접 구현하지 않고 가져와서 보여주기 위함
네이버 블로그의 맥락에서는 서비스에서 공통으로 활용하는 툴과, 실제 게시글 콘텐츠를 분리하기 위함



In [105]:
df_sample=df.iloc[:3,:].copy()

In [106]:
df_sample['content']=df_sample['link'].apply(lambda x: body_text_extractor(x))

블로그 본문을 수집합니다.
9901번째 네이버 블로그 본문 크롤링을 완료했습니다. 
블로그 본문을 수집합니다.
9901번째 네이버 블로그 본문 크롤링을 완료했습니다. 
블로그 본문을 수집합니다.
9901번째 네이버 블로그 본문 크롤링을 완료했습니다. 


In [108]:
df_sample['content_cleaned']=df_sample['content'].apply(lambda x: txtcleaner(x))

In [109]:
df_sample

,title,link,postdate,content,content_cleaned
0,전어 다음은 대하다 월 말 꼭 먹어야 할 가을 제철 먹거리,https://blog.naver.com/leolove100/224419945013,20260922,​\n가을 여행에서 빼놓을 수 없는 게 바로 먹거리다.\n9월 초부터 전어를 찾아 ...,가을 여행에서 빼놓을 수 없는 게 바로 먹거리다 월 초부터 전어를 찾아 바닷가로 떠...
1,강남 논현동 맛집 산초 간장게장 꽃게 무침,https://blog.naver.com/boyun1013/224419938393,20260922,안녕하세요~\n오늘의 맛집 소개는\n강남구 논현동에 위치한 간장게장 맛집 「산초」를...,안녕하세요 오늘의 맛집 소개는 강남구 논현동에 위치한 간장게장 맛집 산초 를 소개해...
2,가을 제철음식 아무 때나 먹으면 된다 맛있는 시기는 따로 있습니,https://blog.naver.com/pangyoung/224419928925,20260922,"가을이면 전어, 대하, 꽃게부터 떠올리죠.\n그런데 ‘가을 제철음식’이라고 해서\n...",가을이면 전어 대하 꽃게부터 떠올리죠 그런데 가을 제철음식 이라고 해서 월부터 월까...
